# Fresh PMSA evaluation — controlled Kaggle run

## Goal
Create matching, metadata-bearing development and external features, then compare four detector heads with shared splits. This is **not yet an executed result**. Chameleon has already been viewed; call this a reproduction, not a new blinded benchmark. No app deployment or model promotion happens here.

This notebook uses ordinary Python and checked subprocess calls, so it does not depend on missing `!python` prefixes. Start in **smoke** mode. Save a Kaggle notebook version and download the stage archive before ending the session.

## Context & methods
Frozen Google SigLIP plus the project's 66-dimensional forensic features. Four heads: semantic linear, forensic linear, semantic nonlinear, fusion. Separate fitting/validation/real calibration/internal test; Chameleon stays out of head training and primary calibration. Full plan: seeds 0/1/2, 50 maximum neural epochs, 2,000 bootstrap resamples, target FPR 5%. Report all seeds; do not tune against Chameleon.

### Key assumptions
Labels and licenses are supplied by dataset owners and must be reviewed. Exact encoded/decoded hashes do not detect all near-duplicates. Global class balance does not remove source/domain shortcuts. Fresh extraction records provenance; it does not by itself establish robustness or a production guarantee.

## 1. Parameters
Attach the updated source archive as a **private** Kaggle dataset. SOURCE_INPUT accepts the dataset folder, its unpacked source/ folder, or the ZIP itself. Update it if your mount differs. Attach original image datasets, not recovered NPZs. Use a new RUN_ID for every independent run.

In [ ]:
from pathlib import Path
MODE = "smoke"  # change to "full" only after the smoke checks pass
FULL_RUN_CONFIRMED = False
RUN_ID = "fresh-smoke-02"  # change for each run; do not reuse names
SOURCE_INPUT = Path("/kaggle/input/datasets/yashyadav56/pmsa-v3-source")
SOURCE_ROOT = Path("/kaggle/working/pmsa_fresh_source_v2")
DATA = {
    "genimage": Path("/kaggle/input/datasets/yangsangtai/tiny-genimage"),
    "ffhq": Path("/kaggle/input/datasets/arnaud58/flickrfaceshq-dataset-ffhq"),
    "fakefaces": Path("/kaggle/input/datasets/hyperclaw79/fakefaces"),
    "coco": Path("/kaggle/input/datasets/xthink/coco-2017-val-images/val2017"),
    "chameleon": Path("/kaggle/input/datasets/yashyadav56/chameleon-private/Chameleon/test"),
}
assert MODE in {"smoke", "full"}
if MODE == "full" and not FULL_RUN_CONFIRMED:
    raise ValueError("Review smoke results first; set FULL_RUN_CONFIRMED=True explicitly.")
import re
assert re.fullmatch(r"[a-z0-9][a-z0-9_-]*", RUN_ID), "Use a safe run identifier"
if not SOURCE_INPUT.exists():
    raise FileNotFoundError(f"Attach the private updated source dataset and update SOURCE_INPUT: {SOURCE_INPUT}")
for name, directory in DATA.items():
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing raw dataset {name}: {directory}")
for name in ("0_real", "1_fake"):
    if not (DATA["chameleon"] / name).is_dir():
        raise FileNotFoundError(f"Chameleon mount needs {name}/")
print("Selected mode:", MODE, "| raw dataset mounts exist")

## 2. Load and check the source snapshot
Supports ZIP and unpacked Kaggle inputs. Only files listed in source_manifest.json are copied after hash checks. Hashes detect changes, not publisher authenticity; use this project's trusted handoff. Existing workspaces/runs are never overwritten. Replace old packages lacking the source manifest with the updated package.

In [ ]:
"""Standard-library source handoff checks, also embedded in the Kaggle notebook.

Hashes provide integrity, not publisher authenticity. Use only trusted sources.
"""
import hashlib
import json
from pathlib import Path, PurePosixPath
import stat
import zipfile


def safe_source_name(name):
    if (not isinstance(name, str) or not name or "\\" in name
            or PurePosixPath(name).is_absolute()
            or any(p in {"", ".", ".."} for p in name.split("/"))):
        raise ValueError("Unsafe source filename")
    return name


def load_source_snapshot(source_input):
    source_input = Path(source_input)
    archive_handle = None
    if source_input.is_dir():
        candidates = [p for p in (source_input, source_input / "source")
                      if (p / "source_manifest.json").is_file() and (p / "pyproject.toml").is_file()]
        if len(candidates) > 1:
            raise ValueError("Ambiguous source folders; choose the exact project folder")
        if not candidates:
            archive = source_input / "pmsa-kaggle-source.zip"
            if archive.is_file():
                return load_source_snapshot(archive)
            raise FileNotFoundError("Updated source manifest missing; upload the updated handoff package")
        root = candidates[0].resolve()

        def reader(name):
            name = safe_source_name(name)
            path = root / name
            if (not path.resolve().is_relative_to(root)
                    or any((root / Path(*Path(name).parts[:i])).is_symlink()
                           for i in range(1, len(Path(name).parts)+1))):
                raise ValueError("Unsafe source symlink")
            if not path.is_file() or path.stat().st_size > 50_000_000:
                raise ValueError("Missing or oversized source file")
            return path.read_bytes()

        manifest = json.loads(reader("source_manifest.json"))
    else:
        archive_handle = zipfile.ZipFile(source_input)
        try:
            entries = archive_handle.infolist()
            if len(entries) > 2000 or sum(p.file_size for p in entries) > 50_000_000:
                raise ValueError("Oversized source archive; use the small source handoff")
            if len({p.filename for p in entries}) != len(entries):
                raise ValueError("Duplicate archive members")
            for entry in entries:
                safe_source_name(entry.filename.rstrip("/"))
                if stat.S_ISLNK(entry.external_attr >> 16):
                    raise ValueError("Unsafe archive symlink")
            try:
                manifest = json.loads(archive_handle.read("source/source_manifest.json"))
            except KeyError as exc:
                raise ValueError("Updated source manifest missing; replace the old ZIP") from exc
            reader = lambda name: archive_handle.read("source/"+safe_source_name(name))
        except Exception:
            archive_handle.close()
            raise
    try:
        files = manifest.get("files")
        if (manifest.get("schema_version") != 1 or not isinstance(files, dict)
                or not 0 < len(files) <= 1000 or "pyproject.toml" not in files):
            raise ValueError("Invalid source manifest")
        payload, total = {}, 0
        for name, digest in sorted(files.items()):
            safe_source_name(name)
            if (not isinstance(digest, str) or len(digest) != 64
                    or any(c not in "0123456789abcdef" for c in digest)):
                raise ValueError("Invalid source digest")
            data = reader(name)
            total += len(data)
            if total > 50_000_000:
                raise ValueError("Oversized source snapshot")
            if hashlib.sha256(data).hexdigest() != digest:
                raise ValueError(f"Source integrity mismatch: {name}")
            payload[name] = data
        identity = hashlib.sha256(json.dumps(files, sort_keys=True,
            separators=(",", ":")).encode()).hexdigest()
        return payload, identity
    finally:
        if archive_handle is not None:
            archive_handle.close()


def prepare_source_snapshot(source_input, workspace):
    payload, digest = load_source_snapshot(source_input)
    workspace = Path(workspace)
    root, marker = workspace / "source", workspace / "source-snapshot-sha256.txt"
    if workspace.exists():
        if not marker.is_file() or marker.read_text().strip() != digest:
            raise FileExistsError("Source workspace differs. Choose a NEW SOURCE_ROOT; do not delete old work.")
        for name, data in payload.items():
            target = root / name
            if (not target.resolve().is_relative_to(root.resolve()) or not target.is_file()
                    or target.is_symlink() or target.read_bytes() != data):
                raise ValueError(f"Existing source workspace changed: {name}; choose a NEW SOURCE_ROOT")
    else:
        workspace.mkdir(parents=True)
        for name, data in payload.items():
            target = root / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
        marker.write_text(digest+"\n")
    return root, digest

import os, subprocess, sys
REPO, source_digest = prepare_source_snapshot(SOURCE_INPUT, SOURCE_ROOT)
os.chdir(REPO)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=REPO, check=True)
RUN = REPO / "outputs" / RUN_ID
if RUN.exists():
    raise FileExistsError("Run already exists. Restore/review it or choose a NEW RUN_ID; do not overwrite.")
RUN.mkdir(parents=True)
print("Source snapshot:", source_digest, "| output:", RUN)

## 3. Environment and GPU check
This uses the notebook's existing compatible Torch installation; it does not reinstall GPU wheels. Extraction uses **one** GPU. T4×2 does not automatically combine GPU memory or double throughput. The actual CUDA operation must pass, not just `cuda.is_available()`.

In [ ]:
run("-m", "pip", "install", "-e", ".[extract,dev]", "gradio>=6.18,<7", "matplotlib>=3.7")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a compatible Kaggle GPU before extracting features")
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
(torch.ones(8, device="cuda") * 2).sum().item()  # fail before extraction if the wheel is incompatible
run("-m", "pytest", "-q")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True)
(RUN / "environment.txt").write_text(freeze.stdout)
plan = {"mode": MODE, "source_snapshot_sha256": source_digest, "heads": ["siglip_probe", "forensic_probe", "siglip_mlp", "fusion"],
        "seeds": [0] if MODE == "smoke" else [0, 1, 2], "epochs": 2 if MODE == "smoke" else 50,
        "bootstrap_resamples": 30 if MODE == "smoke" else 2000, "alpha": 0.05,
        "selection": "Fixed before extraction; no tuning or seed selection using Chameleon.",
        "scope": "Smoke is pipeline validation only; full is previously-viewed benchmark reproduction."}
(RUN / "plan.json").write_text(json.dumps(plan, indent=2))
CACHE = RUN / "feature_cache"
CACHE.mkdir()
import yaml
config = yaml.safe_load((REPO / "configs/default.yaml").read_text())
config["feature"]["cache_dir"] = str(CACHE)
config["feature"]["batch_size"] = 32
config["feature"]["num_workers"] = 2
CONFIG = RUN / "config.yaml"
CONFIG.write_text(yaml.safe_dump(config))
print(json.dumps(plan, indent=2))

## 4. Deterministic manifests
Sorted file enumeration fixes the previous order-dependent selection. Selection is still a capped subset, not a randomly representative sample of the world. The original domain/class imbalance remains a limitation. Raw image decode failures are fatal, not silently skipped.

In [ ]:
train_manifest, test_manifest = CACHE / "manifest_raw_train.csv", CACHE / "manifest_raw_test.csv"
limits = ["--per-generator", "40", "--per-source", "80", "--per-test", "80"] if MODE == "smoke" else ["--per-generator", "2000", "--per-source", "6000"]
run("scripts/build_manifest.py", "--genimage-root", DATA["genimage"], "--ffhq-dir", DATA["ffhq"],
    "--fakefaces-dir", DATA["fakefaces"], "--coco-dir", DATA["coco"], "--chameleon-dir", DATA["chameleon"],
    "--balance", "--seed", "0", "--out-train", train_manifest, "--out-test", test_manifest, *limits)
AUDIT = RUN / "raw_audit"
run("scripts/13_audit_raw_data.py", "--manifest", f"train:{train_manifest}", f"test:{test_manifest}", "--out", AUDIT)
raw_audit = json.loads((AUDIT / "audit.json").read_text())
print(json.dumps(raw_audit["summary"], indent=2))
print("Review class/domain/format counts. Passing exact checks is not a near-duplicate clearance.")

## 5. Save the audit stage
Every archive is private and excludes raw images. Download the zip **and** its SHA256 sidecar from `/kaggle/working`, or save a notebook version with outputs. A local output zip alone is not durable after session deletion.

In [ ]:
def archive_stage(stage):
    output = Path("/kaggle/working") / f"pmsa-{RUN_ID}-{stage}.zip"
    run("scripts/14_package_kaggle.py", "--run", RUN, "--out", output)
    print("DOWNLOAD:", output, "and", output.with_suffix(".zip.sha256"))
archive_stage("audit")

## 6. Matching extraction
Both roles use the same EXIF/RGB/JPEG-q90 policy, slow SigLIP processor, backbone and package environment. Each cache records the resolved backbone revision and original image hashes. Curated exact/related group IDs survive into training. Strict evaluation refuses any metadata mismatch. JPEG standardization is not removal of all source bias.

In [ ]:
for role in ("train", "test"):
    run("scripts/01_extract_features.py", "--config", CONFIG, "--manifest", AUDIT / f"{role}.csv",
        "--tag", f"{role}_v3", "--backbones", "siglip", "forensic", "--normalize-jpeg",
        "--device", "cuda", "--batch-size", "32", "--num-workers", "2")
from pmsa.features import FeatureSet
from pmsa.features.cache import validate_alignment
from pmsa.artifacts import validate_stream_metadata
for tag in ("train_v3", "test_v3"):
    streams = [FeatureSet.load(CACHE / f"{name}_{tag}.npz") for name in ("siglip", "forensic")]
    validate_alignment(streams)
    validate_stream_metadata({s.backbone: s.metadata for s in streams})
    print(tag, [(s.backbone, s.features.shape) for s in streams])
archive_stage("features")

## 7. Fixed four-head comparison
No backbone training. Head fitting runs on CPU from cached features. Every head and seed has a separate directory and calibrator. Fresh mode rejects legacy caches and checks model bundles. Smoke metrics are not publishable generalization estimates; its small splits/bootstrap are only a functional check.

In [ ]:
EVAL = RUN / "evaluation"
run("scripts/10_evaluate_recovered.py", "--config", CONFIG, "--cache-dir", CACHE,
    "--train-tag", "train_v3", "--test-tag", "test_v3", "--require-metadata",
    "--models", ",".join(plan["heads"]), "--seeds", ",".join(map(str, plan["seeds"])),
    "--epochs", plan["epochs"], "--bootstrap", plan["bootstrap_resamples"],
    "--alpha", plan["alpha"], "--cpu-threads", "4", "--out", EVAL,
    "--reason", "All four heads fixed in notebook plan before extraction; no external tuning")
archive_stage("evaluated")

## Results — inspect every seed
AUC is ranking, not accuracy. The FPR/TPR below use frozen development-real thresholds; the 5% target may fail under shift. Seed spread is not a confidence interval. Inspect per-run bootstrap/Wilson intervals in saved JSON rather than averaging their bounds.

In [ ]:
summary = json.loads((EVAL / "summary.json").read_text())
for model, metrics in summary["models"].items():
    print(f'{model:18s} AUC={metrics["auc"]["mean"]:.4f}  FPR={100*metrics["fpr"]["mean"]:.2f}%  TPR={100*metrics["tpr"]["mean"]:.2f}%')
# A simple per-seed plot retains variation; it does not turn seed variation into a statistical CI.
import matplotlib.pyplot as plt
entries = json.loads((EVAL / "results.json").read_text())
names = plan["heads"]
fig, ax = plt.subplots(figsize=(8, 4))
markers = ["o", "s", "^"]
for j, seed in enumerate(plan["seeds"]):
    values = [next(r["external"]["ranking"]["auc"]["point"] for r in entries if r["model"] == name and r["seed"] == seed) for name in names]
    ax.scatter(range(len(names)), values, marker=markers[j], label=f"Seed {seed}")
ax.set_xticks(range(len(names)), ["Semantic linear", "Forensic linear", "Semantic MLP", "Fusion"])
ax.set_ylim(0, 1)
ax.set_ylabel("Chameleon AUC (ranking)")
ax.set_title(f"{MODE.title()} run: per-seed external ranking")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

## Takeaways and next steps

- If smoke succeeded, save/download its outputs, then make a **new** full RUN_ID and set MODE="full", FULL_RUN_CONFIRMED=True. Run top-to-bottom. Do not reuse smoke caches as a full benchmark.
- Save the full stage archive and notebook version. Send back `evaluation/summary.json`, `raw_audit/audit.json`, and the run archive. They preserve everything needed for head comparisons without re-extracting.
- Fresh matching extraction closes missing-preprocessing provenance; it does **not** make a previously-viewed benchmark blind or remove source shortcuts. Near-duplicate/source review remains open.
- Leave deployment unchanged. Before promotion, compare offline scores with online extraction under the recorded environment, and evaluate independent deployment-domain real images. Target-real calibration must stay separate from testing.
- Keep the 100 GPT-Image and 100 Nano Banana collections separate. Do not relabel fake-only evaluation as AUC/FPR evidence; attach a trusted independent real pool before a two-class frontier claim.
- Do not automatically download new datasets, push files to Git, publish the audit, or deploy a selected seed from this notebook.

**Validation status of this distributed template:** code-cell syntax and safety contracts are checked locally. Full top-to-bottom Kaggle execution requires the raw mounts and compatible GPU above; no extraction/training result is prefilled.